# Stage B ModernBERT on V3 + open clinical notes (Colab GPU)
Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all. Cell 2 asks for `data/colab_bundle.zip`
(built on the laptop). The last cell downloads `stage_b_open_results.zip` (checkpoint + metrics).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, zipfile
from google.colab import files
os.makedirs('/content/daa', exist_ok=True); os.chdir('/content/daa')
if not os.path.exists('colab_bundle.zip'):
    files.upload()  # pick data/colab_bundle.zip
zipfile.ZipFile('colab_bundle.zip').extractall('.')
!ls data

In [ ]:
!pip -q install -e .[ml] "transformers>=4.48,<5"

In [ ]:
%env PYTHONPATH=src
!python scripts/train_v3_experiment.py --device cuda --run-name modernbert_text_only_v3_open --extra-train data/m2_open/train.jsonl --max-epochs 6 --patience 2 --batch-size 32 --output-dir data/m2_results_v3

In [ ]:
!python scripts/eval_stage_b.py --checkpoint data/m2_results_v3/checkpoints/modernbert_text_only_v3_open_fold0_seed42.pt --data data/m2_open/test.jsonl | tee data/m2_results_v3/open_test_eval.jsonl

In [ ]:
!zip -r -0 stage_b_open_results.zip data/m2_results_v3 reports/stage_b_runs
files.download('stage_b_open_results.zip')